# Text-to-SQL Experiment Notebook
This notebook connects to MySQL, extracts schema, saves it as JSON, generates SQL with Gemini, validates it, executes it, and explains the result.

In [ ]:
!pip install -q sqlalchemy pymysql pandas python-dotenv langchain-google-genai

In [1]:
import os
import json
import pandas as pd
from sqlalchemy import create_engine, inspect
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

OPENAI_API_KEY =os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY =os.getenv("GEMINI_API_KEY")


NameError: name 'load_dotenv' is not defined

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Create Geminigpt model
llm = ChatGoogleGenerativeAI(
    model = ChatGoogleGenerativeAI(model='gemini-3.5-flash'),
    temperature=0
)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
# Create ChatGPT model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
from sqlalchemy import create_engine, inspect
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
print("Connected Successfully")

Connected Successfully


In [35]:
inspector = inspect(engine)

schema = {
    "database": DB_NAME,
    "tables": []
}

for table in inspector.get_table_names():
    table_info = {
        "table_name": table,
        "columns": [],
        "primary_keys": [],
        "foreign_keys": []
    }

    for col in inspector.get_columns(table):
        table_info["columns"].append({
            "name": col["name"],
            "type": str(col["type"]),
            "nullable": col["nullable"]
        })

    pk = inspector.get_pk_constraint(table)
    table_info["primary_keys"] = pk.get("constrained_columns", [])

    for fk in inspector.get_foreign_keys(table):
        table_info["foreign_keys"].append({
            "column": fk["constrained_columns"],
            "references_table": fk["referred_table"],
            "references_column": fk["referred_columns"]
        })

    schema["tables"].append(table_info)

print("Schema Extracted")

Schema Extracted


In [36]:
with open("schema.json","w",encoding="utf-8") as f:
    json.dump(schema,f,indent=4)

print("schema.json saved")

schema.json saved


In [ ]:
print(json.dumps(schema,indent=4))

In [38]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-pro",
#     temperature=0
# )

question = input("Ask your question: ")

schema_text = json.dumps(schema, indent=2)

prompt = f'''
You are an expert MySQL developer.

Database Schema:
{schema_text}

Rules:
1. Generate ONLY SQL.
2. Do not explain.
3. Use only available tables and columns.
4. Generate valid MySQL syntax.
5. also remove the ```SQL\n\n```
Question:
{question}

SQL:
'''

response = llm.invoke(prompt)
sql_query = response.content.strip()

print("\nGenerated SQL:\n")
print(sql_query)


Generated SQL:

SELECT COUNT(*) AS total_bookings FROM bookings;


In [39]:
blocked = [
    "DROP","DELETE","UPDATE","ALTER",
    "INSERT","TRUNCATE","CREATE"
]

sql_upper = sql_query.upper()

for keyword in blocked:
    if keyword in sql_upper:
        raise Exception(f"Blocked keyword: {keyword}")

print("SQL Validated")

SQL Validated


In [40]:
sql_upper

'SELECT COUNT(*) AS TOTAL_BOOKINGS FROM BOOKINGS;'

In [41]:
result = pd.read_sql(sql_query, engine)
result

,total_bookings
0,801


In [ ]:
prompt = f'''
User Question:
{question}

SQL Result:
{result.to_markdown(index=False)}

Explain the answer in simple English in 1 line .
'''

answer = llm.invoke(prompt)
print(answer.content)

The SQL result shows that there are a total of 80 drivers. This means that if you were to count all the drivers in the database or system being queried, you would find 80 of them.
